# [Experiment] Parameter-Matched KAN-TabNet | Cosine Annealing LR | Forest Cover

### Overview
This notebook evaluates the parameter-matched KAN-TabNet architecture under the continuous optimization environment of a Cosine Annealing learning rate schedule.

### Methodological Context: The Final Synthesis
This experiment represents the synthesis of our structural modifications (KAN layers replacing standard linear transformations) and our shifted optimization thermodynamics (CosineLR). By comparing these results directly against the Vanilla CosineLR baseline, we maintain strict experimental controls while evaluating the architecture in a modern learning environment that deliberately departs from the original paper's StepLR approach.

### The Hypothesis
We hypothesize that the learnable B-splines inherent to KAN layers may interact uniquely with the smooth, continuous decay of the Cosine schedule. This notebook investigates whether this gradual "cooling" of the learning rate allows the KAN splines to settle into more optimal, stable configurations compared to the sudden, discrete adjustments forced by the traditional StepLR schedule.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier

dataset = fetch_covtype()

X = dataset.data
# CovType is 1-indexed (1 to 7); PyTorch expects 0-indexed labels
y = dataset.target - 1

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"Train shape: {X_train.shape}")
print(f"Valid shape: {X_valid.shape}")
print(f"Test shape:  {X_test.shape}")

Train shape: (464809, 54)
Valid shape: (58101, 54)
Test shape:  (58102, 54)


### Model Training

In [5]:
MAX_EPOCHS = 1000

In [6]:
# Hyperparameters from original paper.
# Note: momentum is 1 - 0.7 (paper m_B) to align with PyTorch's BatchNorm API.
paper_config = {
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-4,
    'momentum': 0.3,
    'optimizer_fn': torch.optim.Adam,
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=20, # adjusted so parameter count is matched with vanilla TabNet
    n_a=20, # adjusted so parameter count is matched with vanilla TabNet
    **paper_config,
    clip_value=2.,
    optimizer_params=dict(lr=0.003), # reduced as 0.02 lr is too aggressive
    scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingLR,
    scheduler_params=dict(T_max=MAX_EPOCHS, eta_min=1e-5),
    use_kan=True,
    kan_grid_size=5,
    kan_spline_order=3,
    seed=seed,
    verbose=50
)

In [7]:
# Hyperparameters from original paper.
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=100,
    num_workers=0,
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 1.31679 | valid_accuracy: 0.36378 |  0:00:09s
epoch 50 | loss: 0.21747 | valid_accuracy: 0.91661 |  0:06:49s
epoch 100| loss: 0.18982 | valid_accuracy: 0.92745 |  0:13:29s
epoch 150| loss: 0.1118  | valid_accuracy: 0.95282 |  0:20:08s
epoch 200| loss: 0.11185 | valid_accuracy: 0.9553  |  0:26:48s
epoch 250| loss: 0.08987 | valid_accuracy: 0.96024 |  0:33:27s
epoch 300| loss: 0.11991 | valid_accuracy: 0.9525  |  0:40:08s
epoch 350| loss: 0.09814 | valid_accuracy: 0.95398 |  0:46:48s
epoch 400| loss: 0.07607 | valid_accuracy: 0.9638  |  0:53:29s
epoch 450| loss: 0.07081 | valid_accuracy: 0.96442 |  1:00:10s
epoch 500| loss: 0.05911 | valid_accuracy: 0.96783 |  1:06:51s
epoch 550| loss: 0.064   | valid_accuracy: 0.96714 |  1:13:31s
epoch 600| loss: 0.05389 | valid_accuracy: 0.96904 |  1:20:12s
epoch 650| loss: 0.05487 | valid_accuracy: 0.96874 |  1:26:54s
epoch 700| loss: 0.04816 | valid_accuracy: 0.96945 |  1:33:35s
epoch 750| loss: 0.05718 | valid_accuracy: 0.9674  |  1

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 469,336


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.97258


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/forest_cover/tables'
MODELS_DIR = f'{BASE_DIR}/results/forest_cover/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Forest Cover",
    "parameters": param_count,
    "scheduler": "CosineAnnealingLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_kan.best_epoch,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/04_kan_param_matched_cosine_lr_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/04_kan_param_matched_cosine_lr_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/forest_cover/tables/04_kan_param_matched_cosine_lr_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/forest_cover/models/04_kan_param_matched_cosine_lr_469k.zip
